In [31]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [33]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
import json
from pydoc import text


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "python" or "json" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code
* Give only 3 items, 1 by each type (Python, JSON, Regex)
* Respond ONLY the JSON Object, no markdown, no backticks, no explanation.
"""
    messages = []
    add_user_message(messages, prompt)
    text = chat(messages)
    return json.loads(text)

In [ ]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent = 2)

In [105]:
print(json.dumps(dataset, indent = 2))

[
  {
    "task": "Write a Python function that parses an AWS CloudFormation template in JSON format and returns a list of all resource logical IDs that have a Type of 'AWS::S3::Bucket'",
    "format": "python",
    "solution_criteria": "Function must correctly identify and return all S3 bucket resource logical IDs from a CloudFormation template, handling nested structures appropriately"
  },
  {
    "task": "Create a JSON object representing an AWS IAM policy that allows read-only access to all objects in an S3 bucket named 'my-data-bucket' and denies access to objects with the prefix 'private/'",
    "format": "json",
    "solution_criteria": "JSON must be a valid IAM policy with correct Statement structure, Actions, Resources, and Effect values that accurately represent the read-only and deny requirements"
  },
  {
    "task": "Write a regular expression that matches valid AWS S3 bucket names according to AWS naming rules (3-63 characters, lowercase letters, numbers, hyphens, no con

In [106]:
def run_prompt(test_case):
  """Merges the prompt and test case input, then return the results"""
  prompt = f"""
  Solve the following task:

  {test_case['task']}

  * Response only with Python, JSON, or a plain Regex; without any markdown, backticks, or explanation.
  * Do not add any comments or commentary or explanation.
  """
  
  messages = []
  add_user_message(messages, prompt)
  # add_assistant_message(messages, "```code")
  output = chat(messages)

  return output

In [107]:
def grade_by_model(test_case, output):
  # Create evaluation prompt
  eval_prompt = f"""
  You are an expert code reviewer. Evaluate this AI-generated solution.
  
  Original Task:
  <task>
  {test_case['task']}
  </task>

  Solution to Evaluate: 
  <solution>
  {output}
  </solution>

  Criteria you should use to evaluate the solution:
  <criteria>
  {test_case['solution_criteria']}
  </criteria>
  
  Output Format:
  Provide your evaluation as a structured JSON object with:
  - "strengths": An array of 1-3 key strengths
  - "weaknesses": An array of 1-3 key areas for improvement  
  - "reasoning": A concise explanation of your assessment
  - "score": A number between 1-10
  Respond ONLY with the JSON, no markdown, no backticks, no explanation.

  Keep your response concise and direct.
  Example response shape:
  {{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
  }}
  """

  messages = []
  
  add_user_message(messages, eval_prompt)
  add_assistant_message(messages,"{")
  
  eval_text = chat(messages)

  return json.loads("{" + eval_text)

In [79]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    
def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    elif format == "regex":
        return validate_regex(response)
    else:
        raise ValueError(f"Unknown format: {format}")

In [88]:
import re

def strip_code_blocks(text: str) -> str:
  pattern = r'^```[\w]*\n?(.*?)```$'
  match = re.search(pattern, text.strip(), re.DOTALL)
  if match:
      return match.group(1).strip()
  return text.strip()

def run_test_case(test_case):
  """Calls run_prompt, then grades the results"""
  output = run_prompt(test_case)
  output = strip_code_blocks(output)

  model_grade = grade_by_model(test_case, output)
  model_score = model_grade["score"]
  reasoning = model_grade["reasoning"]

  syntax_score = grade_syntax(output, test_case)

  score = (model_score + syntax_score) / 2

  return {
    "output": output,
    "test_case": test_case,
    "score": score,
    "reasoning": reasoning
  }

In [81]:
def run_eval(dataset):
  """Loads the dataset and calls run test case with each case"""
  results = []

  for test_case in dataset:
    result = run_test_case(test_case)
    results.append(result)

  return results

In [108]:
with open("dataset.json", "r") as f:
  dataset = json.load(f)

results = run_eval(dataset)

In [109]:
print(json.dumps(results, indent = 2))

[
  {
    "output": "import json\nimport re\n\ndef get_s3_bucket_logical_ids(template_json):\n    \"\"\"\n    Parse an AWS CloudFormation template and return logical IDs of S3 buckets.\n    \n    Args:\n        template_json: JSON string of CloudFormation template\n    \n    Returns:\n        List of logical IDs for AWS::S3::Bucket resources\n    \"\"\"\n    template = json.loads(template_json)\n    s3_bucket_ids = []\n    \n    resources = template.get('Resources', {})\n    \n    for logical_id, resource_definition in resources.items():\n        if resource_definition.get('Type') == 'AWS::S3::Bucket':\n            s3_bucket_ids.append(logical_id)\n    \n    return s3_bucket_ids\n\n\nif __name__ == \"__main__\":\n    sample_template = \"\"\"{\n        \"AWSTemplateFormatVersion\": \"2010-09-09\",\n        \"Resources\": {\n            \"MyFirstBucket\": {\n                \"Type\": \"AWS::S3::Bucket\",\n                \"Properties\": {\n                    \"BucketName\": \"my-first-b